# BGL Streaming Evaluation

This notebook demonstrates how to run the streaming pipeline on the BGL labelled dataset and assess detection quality against the built-in alert tags (non-`-` labels indicate alerts).

In [ ]:
from pathlib import Path
import polars as pl
import numpy as np
from sklearn.metrics import precision_recall_curve, auc, precision_score, recall_score, f1_score

from log_anomaly_analysis.core.streaming_pipeline import (
    StreamingPipeline,
    StreamingPipelineState,
    StreamingResultWriter,
    stream_file_dataset,
)
from log_anomaly_analysis.core.components.preprocessing import PreprocessorComponent
from log_anomaly_analysis.datasets.loaders import load_dataset


In [ ]:
REPO_ROOT = Path.cwd() / ".." / ".."
CONFIG_PATH = REPO_ROOT / "configs" / "labeled" / "BGL.yaml"
CHUNK_SIZE = 50_000


In [ ]:
pipeline = StreamingPipeline(str(CONFIG_PATH))
writer = StreamingResultWriter(pipeline.output_dir, pipeline.config.output.formats)
state = pipeline.process_stream(
    stream_file_dataset(pipeline.config, chunk_size=CHUNK_SIZE),
    state=StreamingPipelineState(),
    writer=writer,
)
pipeline.flush_visualizations(state)
output_dir = pipeline.output_dir
print(f"Results saved to: {output_dir}")
print(f"Chunks processed: {state.chunk_index}")
print(f"Total logs: {state.total_logs}")
print(f"Total windows: {state.total_windows}")
print(f"Total detected anomalies: {state.total_anomalies}")


In [ ]:
def load_parquet_stage(stage: str) -> pl.DataFrame:
    stage_dir = output_dir / stage
    if not stage_dir.exists():
        return pl.DataFrame()
    files = sorted(stage_dir.glob("*.parquet"))
    if not files:
        return pl.DataFrame()
    return pl.concat([pl.read_parquet(file) for file in files], how="vertical_relaxed")

anomalies_df = load_parquet_stage("anomalies")
windowed_df = load_parquet_stage("windowed")
event_matrix_df = load_parquet_stage("event_matrix")
anomalies_df.head()


In [ ]:
raw_logs = load_dataset(pipeline.config.dataset.__dict__)
preprocessor = PreprocessorComponent(pipeline.config.preprocessing.__dict__)
preprocessed = preprocessor.process(raw_logs)
window_size = pipeline.config.windowing.window_size or "5m"
ground_truth = (
    preprocessed
    .with_columns(pl.col("Timestamp").dt.truncate(window_size).alias("Window"))
    .group_by("Window")
    .agg(
        [
            (pl.col("Label") != "-").any().alias("HasAlert"),
            pl.len().alias("WindowLogCount"),
        ]
    )
)
ground_truth.head()


In [ ]:
if anomalies_df.is_empty():
    print("No anomaly records found; metrics cannot be computed.")
else:
    joined = (
        anomalies_df
        .join(windowed_df.select("Window"), on="Window", how="left")
        .join(ground_truth, on="Window", how="left")
        .drop_nulls("HasAlert")
    )
    y_true = joined["HasAlert"].to_numpy().astype(bool)
    scores = joined["AnomalyScore"].to_numpy()
    y_pred = joined["IsAnomaly"].to_numpy().astype(bool)

    precision, recall, _ = precision_recall_curve(y_true, scores)
    pr_auc = auc(recall, precision) if len(recall) > 1 else float('nan')
    precision_at_threshold = precision_score(y_true, y_pred, zero_division=0)
    recall_at_threshold = recall_score(y_true, y_pred, zero_division=0)
    f1_at_threshold = f1_score(y_true, y_pred, zero_division=0)

    summary = pl.DataFrame(
        {
            "Metric": ["PR-AUC", "Precision", "Recall", "F1"],
            "Value": [pr_auc, precision_at_threshold, recall_at_threshold, f1_at_threshold],
        }
    )
    display(summary.to_pandas())

    print(f"Total evaluated windows: {len(joined)}")
    print(f"Positive rate: {y_true.mean():.3%}")
